load documents

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# PDF Path
pdf_path = "D:/agentic_rag_2/scholar_ship_details.pdf"

# Load PDF
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Print Results
print(f"Total Pages: {len(documents)}")

print("\nFirst Page Content:\n")
print(documents[0].page_content[:1000])

Total Pages: 6

First Page Content:

Tamil Nadu and India Scholarship Guide 2026
RAG Chatbot Knowledge Base
Complete Reference for Student Financial Support Schemes
Central Government | Tamil Nadu State | Special Categories
Scholarship Discovery | Eligibility Rules | Application Guidance
INTRODUCTION
Education Affordability in India
Education plays a crucial role in improving employment opportunities, economic growth, and social mobility. However,
many students in India face financial challenges that prevent them from pursuing higher education. Rising tuition fees,
hostel expenses, examination fees, transportation costs, books, and digital learning requirements create barriers for
students from economically weaker backgrounds.
Why Students Miss Scholarship Opportunities
Despite numerous scholarships offered by Central Government, State Governments, educational institutions, and
private organizations, many students remain unaware. Common reasons include:
- Lack of awareness about availa

text splitter

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200
)

## Split the Documents
chunks = text_splitter.split_documents(documents)

## Print Result
print(f"Total Chunks: {len(chunks)}")

print("\nFirst Chunk:\n")
print(chunks[0].page_content)

print("\nMetadata:\n")
print(chunks[0].metadata)

Total Chunks: 39

First Chunk:

Tamil Nadu and India Scholarship Guide 2026
RAG Chatbot Knowledge Base
Complete Reference for Student Financial Support Schemes
Central Government | Tamil Nadu State | Special Categories
Scholarship Discovery | Eligibility Rules | Application Guidance
INTRODUCTION
Education Affordability in India
Education plays a crucial role in improving employment opportunities, economic growth, and social mobility. However,

Metadata:

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-24T01:48:48+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-24T01:48:48+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'D:/agentic_rag_2/scholar_ship_details.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


Embeddings

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

vectordb store

In [5]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


Similarity search


In [6]:
# User Question
question = "What are the eligibility criteria for this scholarship??"

# Similarity Search
retrieved_docs = vector_store.similarity_search(
    query=question,
    k=3
)

# Print Results
print(f"Retrieved Documents: {len(retrieved_docs)}\n")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"Document {i}")
    print("-" * 50)
    print(doc.page_content)
    print()

Retrieved Documents: 3

Document 1
--------------------------------------------------
Eligibility: Students with disabilities enrolled in technical courses.
Income Limit: Family income up to Rs.8 lakh.
Benefits: Rs.50,000 annually.
Application: Online application through NSP.
5. INSPIRE Scholarship
Objective: Promote science education and research.
Eligibility: Top-performing students pursuing science streams.
Income Limit: Merit-based (no strict income limit).
Benefits: Scholarship and mentorship support.
Application: Apply through INSPIRE portal.

Document 2
--------------------------------------------------
Eligibility: Students with disabilities enrolled in technical courses.
Income Limit: Family income up to Rs.8 lakh.
Benefits: Rs.50,000 annually.
Application: Online application through NSP.
5. INSPIRE Scholarship
Objective: Promote science education and research.
Eligibility: Top-performing students pursuing science streams.
Income Limit: Merit-based (no strict income limit).
Be

Basic RAG

In [7]:
## Load API KEY

from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Get Groq API Key
groq_api_key = os.getenv("GROQ_API_KEY")

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Prompt
RAG_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful AI assistant. Answer the user's question using only the provided context. If the answer is not in the context, say 'I don't know.'"
    ),
    (
        "human",
        "Context:\n{context}\n\nQuestion:\n{question}"
    ),
])

# User Question
question = "What are the eligibility criteria for this scholarship?"

# Retrieve Top 3 Documents
retrieved_docs = vector_store.similarity_search(
    query=question,
    k=3
)

# Create Context
context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

# RAG Chain
chain = RAG_PROMPT | llm | StrOutputParser()

# Generate Answer
answer = chain.invoke({
    "context": context,
    "question": question
})

print("Question:\n", question)
print("\nAnswer:\n", answer)

Question:
 What are the eligibility criteria for this scholarship?

Answer:
 There are two scholarships mentioned. 

For the first scholarship (no specific name mentioned), the eligibility criteria are: 
1. Students with disabilities 
2. Enrolled in technical courses

For the INSPIRE Scholarship, the eligibility criteria are: 
1. Top-performing students 
2. Pursuing science streams.


Langgraph Basic

In [9]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# State
class RAGState(TypedDict):
    question: str
    documents: list
    answer: str


# Node 1
def retrieve(state: RAGState):
    print("Retrieve Node")
    return {
        "documents": ["Document 1", "Document 2", "Document 3"]
    }


# Node 2
def generate(state: RAGState):
    print("Generate Node")
    return {
        "answer": "This is the generated answer."
    }


# Build Graph
builder = StateGraph(RAGState)

builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

graph = builder.compile()


# Run Graph
result = graph.invoke({
    "question": "What scholarships are available for engineering students?"
})

print(result)

Retrieve Node
Generate Node
{'question': 'What scholarships are available for engineering students?', 'documents': ['Document 1', 'Document 2', 'Document 3'], 'answer': 'This is the generated answer.'}


Agentic RAG

In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

# State
class AgentState(TypedDict):
    question: str
    documents: list
    answer: str

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Retrieve Node
def retrieve(state: AgentState):
    docs = vector_store.similarity_search(
        state["question"],
        k=3
    )

    return {
        "documents": docs
    }

# Generate Node
def generate(state: AgentState):
    context = "\n\n".join(
        doc.page_content for doc in state["documents"]
    )

    prompt = f"""
You are a helpful AI assistant.

Answer using only the provided context.

Context:
{context}

Question:
{state["question"]}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

# Build Graph
builder = StateGraph(AgentState)

builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

agentic_rag = builder.compile()

# Run
result = agentic_rag.invoke({
    "question": "What scholarships are available for engineering students?"
})

print(result["answer"])

The provided context does not specifically mention scholarships available for engineering students. However, it is mentioned that you are a first-generation engineering student, and you can apply for schemes, but the specific schemes are not mentioned in the given context.


Conditional Edges:

```mermaid
flowchart TD

    A[START]
    B[Retrieve Documents]
    C{Documents Found and Think?}
    D[Generate Answer]
    E[END]

    A --> B
    B --> C
    C -->|Yes| D
    C -->|No| E
    D --> E
```

Normal edge :
```mermaid
flowchart TD

    A[START]
    B[Retrieve Documents]
    C[Generate Answer]
    D[END]

    A --> B
    B --> C
    C --> D
```

> **🧠 `decide()` Function**
>
> `Retrieve` node முடிந்த பிறகு, **"Documents கிடைத்ததா?"** என்று check பண்ணும்.
>
> - ✅ Documents கிடைத்தால் → **Generate** node-க்கு அனுப்பும்.
> - ❌ Documents கிடைக்கவில்லை என்றால் → **END**-க்கு அனுப்பும்.
>
> **அதாவது, `decide()` function-ன் வேலை அடுத்த எந்த node-க்கு செல்ல வேண்டும் என்று முடிவு (Decision) எடுப்பது.**

In [11]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# State
class AgentState(TypedDict):
    question: str
    documents: list
    answer: str


# Retrieve Node
def retrieve(state: AgentState):
    docs = vector_store.similarity_search(state["question"], k=3)
    return {"documents": docs}


# Decide Node (Conditional)
def decide(state: AgentState) -> Literal["generate", "__end__"]:
    if len(state["documents"]) == 0:
        return END
    return "generate"


# Generate Node
def generate(state: AgentState):
    context = "\n\n".join(doc.page_content for doc in state["documents"])

    response = llm.invoke(f"""
Context:
{context}

Question:
{state["question"]}
""")

    return {"answer": response.content}


# Build Graph
builder = StateGraph(AgentState)

builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)


## Conditional Edge 
builder.add_edge(START, "retrieve")
builder.add_conditional_edges(
    "retrieve",
    decide,
    {
        "generate": "generate",
        END: END
    }
)

builder.add_edge("generate", END)

graph = builder.compile()


# Run
result = graph.invoke({
    "question": "What scholarships are available for engineering students?"
})

print(result)

{'question': 'What scholarships are available for engineering students?', 'documents': [Document(id='970ae741-6a1f-463c-b10d-482d126f509f', metadata={'trapped': '/False', 'creator': '(unspecified)', 'page_label': '1', 'creationdate': '2026-06-24T01:48:48+00:00', 'moddate': '2026-06-24T01:48:48+00:00', 'subject': '(unspecified)', 'keywords': '', 'source': 'D:/agentic_rag_2/scholar_ship_details.pdf', 'producer': 'ReportLab PDF Library - (opensource)', 'author': '(anonymous)', 'page': 0, 'title': '(anonymous)', 'total_pages': 6}, page_content='- What scholarships are available for SC students?\n- I am a first-generation engineering student. Which schemes can I apply for?\n- What scholarships are available for female students?\n- How do I apply through NSP?\n- What documents are required for BC scholarship?\nCENTRAL GOVERNMENT SCHOLARSHIPS\n1. Central Sector Scheme of Scholarships (CSSS)\nObjective: Support meritorious students from economically weaker sections.'), Document(id='8c22d2eb-27

Query Rewriting

> **Query Rewriting என்பது, User கேட்ட கேள்வியை Search Engine அல்லது RAG System எளிதாக புரிந்துகொள்ளும் வகையில் AI தெளிவாக மாற்றி எழுதுவது.**

---

## Workflow

```mermaid
flowchart TD

    A[User Question]
    B[Query Rewriting]
    C[Improved Question]
    D[Retrieve Documents]
    E[Generate Answer]

    A --> B
    B --> C
    C --> D
    D --> E
```

---

## Example

### User Question

```text
First graduate?
```

⬇️

### Rewritten Query

```text
What is the eligibility criteria for the First Graduate Scholarship?
```

---

## Why Query Rewriting?

- Makes the user's question more clear.
- Improves document retrieval.
- Helps the RAG system find more relevant documents.
- Produces a more accurate answer.

In [12]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

# State
class AgentState(TypedDict):
    question: str
    rewritten_question: str
    documents: list
    answer: str

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Query Rewrite Node
def rewrite_query(state: AgentState):
    prompt = f"""
Rewrite the following user question to improve retrieval.
Return only the rewritten question.

Question:
{state["question"]}
"""

    response = llm.invoke(prompt)

    return {
        "rewritten_question": response.content
    }

# Retrieve Node
def retrieve(state: AgentState):
    docs = vector_store.similarity_search(
        state["rewritten_question"],
        k=3
    )

    return {
        "documents": docs
    }

# Generate Node
def generate(state: AgentState):
    context = "\n\n".join(
        doc.page_content for doc in state["documents"]
    )

    prompt = f"""
Context:
{context}

Question:
{state["rewritten_question"]}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

# Graph
builder = StateGraph(AgentState)

builder.add_node("rewrite_query", rewrite_query)
builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)

builder.add_edge(START, "rewrite_query")
builder.add_edge("rewrite_query", "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

graph = builder.compile()

# Run
result = graph.invoke({
    "question": "Engineering scholarship?" ## sample question
})

print(result["rewritten_question"])
print(result["answer"])

What are some available engineering scholarships and how can I apply for them?
Based on the provided context, here are some available engineering scholarships and how you can apply for them:

1. **Central Sector Scheme of Scholarships (CSSS)**: This scheme supports meritorious students from economically weaker sections. Although the context doesn't explicitly mention engineering, it's a central government scholarship that you can consider.

As a first-generation engineering student, you can apply for the CSSS scheme. To apply, you'll need to go through the National Scholarship Portal (NSP). Here's a general outline of the application process:

* Visit the NSP website and register yourself.
* Fill out the application form and upload the required documents.
* Submit the application and take a printout of the confirmation page.

Some other scholarships that might be available for engineering students include:

* **SC scholarships**: Although not explicitly mentioned in the context, there 

document grader = filter documents

```mermaid
flowchart TD

    A[Question]
    B[Retrieve 3 Documents]
    C[Document Grader]
    D[Keep Relevant Documents]
    E[Generate Answer]

    A --> B
    B --> C
    C --> D
    D --> E
```

In [13]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# State
class AgentState(TypedDict):
    question: str
    documents: list
    filtered_documents: list
    answer: str


# Retrieve Node
def retrieve(state: AgentState):
    docs = vector_store.similarity_search(
        state["question"],
        k=3
    )
    return {"documents": docs}


# Document Grader Node
def grade_documents(state: AgentState):
    filtered_docs = []

    for doc in state["documents"]:

        prompt = f"""
You are a document grader.

Question:
{state["question"]}

Document:
{doc.page_content}

Is this document relevant to the question?

Answer only YES or NO.
"""

        response = llm.invoke(prompt)

        if "YES" in response.content.upper():
            filtered_docs.append(doc)

    return {"filtered_documents": filtered_docs}


# Generate Node
def generate(state: AgentState):

    context = "\n\n".join(
        doc.page_content for doc in state["filtered_documents"]
    )

    response = llm.invoke(f"""
Context:
{context}

Question:
{state["question"]}
""")

    return {"answer": response.content}


# Build Graph
builder = StateGraph(AgentState)

builder.add_node("retrieve", retrieve)
builder.add_node("grade_documents", grade_documents)
builder.add_node("generate", generate)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade_documents")
builder.add_edge("grade_documents", "generate")
builder.add_edge("generate", END)

graph = builder.compile()


# Test
result = graph.invoke({
    "question": "What documents are required for the First Graduate Scholarship?"
})


# Retrieved Documents
print("\nRetrieved Documents")
print("=" * 60)

for i, doc in enumerate(result["documents"], 1):
    print(f"\nDocument {i}")
    print("-" * 50)
    print(doc.page_content)


# Filtered Documents
print("\n\nRelevant Documents")
print("=" * 60)

for i, doc in enumerate(result["filtered_documents"], 1):
    print(f"\nDocument {i}")
    print("-" * 50)
    print(doc.page_content)


# Final Answer
print("\n\nFinal Answer")
print("=" * 60)
print(result["answer"])


Retrieved Documents

Document 1
--------------------------------------------------
conditions. Some schemes may have restrictions on combining benefits.
Q: What documents are commonly required?
A: Most scholarships require: Aadhaar Card, Income Certificate, Community Certificate, Bonafide Certificate, Mark
Sheets, Bank Account Details, and a Passport Size Photograph. Specific schemes may need additional documents.
Q: How can students track application status?
A: Students can track their application status through the respective scholarship portal - either the National

Document 2
--------------------------------------------------
conditions. Some schemes may have restrictions on combining benefits.
Q: What documents are commonly required?
A: Most scholarships require: Aadhaar Card, Income Certificate, Community Certificate, Bonafide Certificate, Mark
Sheets, Bank Account Details, and a Passport Size Photograph. Specific schemes may need additional documents.
Q: How can students track 

# Web Search

> **Web Search (இணைய தேடல்)** என்பது, இணையத்தில் (Internet) இருந்து பயனரின் கேள்விக்கு தொடர்புடைய தகவல்களை தேடி எடுத்து வருவது.

---

## Workflow

```mermaid
flowchart TD

    A[User Question]
    B[Tavily Web Search]
    C[Relevant Web Results]
    D[LLM<br/>Groq / OpenAI / Claude]
    E[Final Answer]

    A --> B
    B --> C
    C --> D
    D --> E
```

---

## Other Alternatives to Tavily

- Google Search API
- Bing Search API
- Brave Search API
- Serper API
- SerpAPI
- Exa AI Search
- Firecrawl

---

## Example

**User Question**

```text
What are the latest AI trends in 2026?
```

↓

**Tavily Web Search**

```text
Searches the Internet
```

↓

**Relevant Web Results**

```text
Article 1
Article 2
Article 3
```

↓

**LLM**

```text
Reads the search results and generates the final answer.
```

In [14]:
## Load API KEY
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Get Groq API Key
tavily_api_key = os.getenv("TAVILY_API_KEY")

In [15]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

# State
class AgentState(TypedDict):
    question: str
    web_results: str
    answer: str

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Web Search Tool
web_search = TavilySearch(max_results=3)

# Web Search Node
def web_search_node(state: AgentState):

    results = web_search.invoke(state["question"])

    return {
        "web_results": str(results)
    }

# Generate Node
def generate(state: AgentState):

    response = llm.invoke(f"""
Use the following web search results to answer the question.

Web Results:
{state["web_results"]}

Question:
{state["question"]}
""")

    return {
        "answer": response.content
    }

# Build Graph
builder = StateGraph(AgentState)

builder.add_node("web_search", web_search_node)
builder.add_node("generate", generate)

builder.add_edge(START, "web_search")
builder.add_edge("web_search", "generate")
builder.add_edge("generate", END)

graph = builder.compile()

# Run
result = graph.invoke({
    "question": "Latest Scholarship for Arts Stutends ?"
})

print(result["answer"])

According to the search results, here are some latest scholarships for arts students:

1. **5563+ Art & Art studies scholarships 2026-27**: This scholarship is available for international students to study abroad, with eligibility criteria, deadlines, application form, and selection process available on the website [https://www.wemakescholars.com/art-art-studies-scholarships-for-international-students-to-study-abroad](https://www.wemakescholars.com/art-art-studies-scholarships-for-international-students-to-study-abroad).
2. **Top 120 Art Scholarships with July 2026 Deadlines**: This scholarship is available for art students, with awards available for various mediums such as oil paint, pottery, or 3D animation, and can be found on the website [https://bold.org/scholarships/by-major/art-scholarships](https://bold.org/scholarships/by-major/art-scholarships).
3. **Art scholarships around the world**: There are numerous scholarships available for art students around the world, including fun

# Router Agent

> **Router Agent** என்பது, User கேள்வியை பார்த்து **எந்த Source-லிருந்து Answer எடுக்க வேண்டும்** என்று முடிவு செய்யும் AI Agent.

```mermaid
flowchart TD

    A[User Question]
    B[Router Agent]
    C[Vector Database]
    D[Web Search]
    E[Final Answer]

    A --> B
    B -->|Vector| C
    B -->|Web| D
    C --> E
    D --> E
```

## Example

**Question**

```text
What is the eligibility criteria for the First Graduate Scholarship?
```

➡️ **Route:** Vector Database

---

**Question**

```text
What are the latest AI trends in 2026?
```

➡️ **Route:** Web Search

In [16]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

# State
class AgentState(TypedDict):
    question: str
    datasource: str
    answer: str

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Router Node
def router(state: AgentState):

    prompt = f"""
You are a router.

If the question can be answered from the Vector Database,
return "vector".

If the question requires recent or latest information,
return "web".

Question:
{state["question"]}

Answer only: vector or web.
"""

    response = llm.invoke(prompt)

    return {
        "datasource": response.content.strip().lower()
    }


# Vector Search Node
def vector_search(state: AgentState):
    return {
        "answer": "Answer from Vector Database."
    }


# Web Search Node
def web_search(state: AgentState):
    return {
        "answer": "Answer from Web Search."
    }


# Route Decision
def route(state: AgentState) -> Literal["vector_search", "web_search"]:

    if state["datasource"] == "web":
        return "web_search"

    return "vector_search"


# Build Graph
builder = StateGraph(AgentState)

builder.add_node("router", router)
builder.add_node("vector_search", vector_search)
builder.add_node("web_search", web_search)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    route,
    {
        "vector_search": "vector_search",
        "web_search": "web_search"
    }
)

builder.add_edge("vector_search", END)
builder.add_edge("web_search", END)

graph = builder.compile()


# Run
result = graph.invoke({
    "question": "What are the free sholorships in 2026?"
})

print(result)

{'question': 'What are the free sholorships in 2026?', 'datasource': 'web', 'answer': 'Answer from Web Search.'}


# Hallucination Checker

> **Hallucination Checker** என்பது, LLM உருவாக்கிய Answer உண்மையாக Documents-ல் இருக்கிறதா அல்லது AI தானாக உருவாக்கிய தகவலா என்பதை சரிபார்க்கும் Agent.

```mermaid
flowchart TD

    A[Question]
    B[Retrieve Documents]
    C[Generate Answer]
    D[Hallucination Checker]
    E[Grounded?]
    F[Final Answer]

    A --> B
    B --> C
    C --> D
    D --> E
    E -->|YES| F
    E -->|NO| B
```

## Example

**Context**

```text
Scholarship amount: ₹50,000
```

↓

**LLM Answer**

```text
Scholarship amount: ₹50,000
```

✅ Grounded = YES

---

**LLM Answer**

```text
Scholarship amount: ₹75,000
```

❌ Grounded = NO

In [17]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

# State
class AgentState(TypedDict):
    question: str
    documents: list
    answer: str
    grounded: str

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Retrieve Node
def retrieve(state: AgentState):
    docs = vector_store.similarity_search(
        state["question"],
        k=3
    )
    return {"documents": docs}

# Generate Node
def generate(state: AgentState):

    context = "\n\n".join(
        doc.page_content for doc in state["documents"]
    )

    response = llm.invoke(f"""
Context:
{context}

Question:
{state["question"]}
""")

    return {
        "answer": response.content
    }

# Hallucination Checker
def hallucination_checker(state: AgentState):

    context = "\n\n".join(
        doc.page_content for doc in state["documents"]
    )

    prompt = f"""
You are a hallucination checker.

Context:
{context}

Answer:
{state["answer"]}

Is the answer completely supported by the context?

Answer only YES or NO.
"""

    response = llm.invoke(prompt)

    return {
        "grounded": response.content.strip()
    }

# Build Graph
builder = StateGraph(AgentState)

builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)
builder.add_node("hallucination_checker", hallucination_checker)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", "hallucination_checker")
builder.add_edge("hallucination_checker", END)

graph = builder.compile()

# Run
result = graph.invoke({
    "question": "What documents are required for the First Graduate Scholarship?"
})

print("Answer:\n", result["answer"])
print("\nHallucination Check:", result["grounded"])

Answer:
 Most scholarships, including the First Graduate Scholarship, require the following documents: 

1. Aadhaar Card
2. Income Certificate
3. Community Certificate
4. Bonafide Certificate
5. Mark Sheets
6. Bank Account Details
7. Passport Size Photograph

Please note that specific schemes may need additional documents, so it's best to check the respective scholarship portal for the most accurate and up-to-date information.

Hallucination Check: YES


# Answer Grader

> **Answer Grader** என்பது, LLM உருவாக்கிய Answer, User கேட்ட Question-க்கு சரியாக பதில் அளிக்கிறதா என்பதை சரிபார்க்கும் Agent.

```mermaid
flowchart TD

    A[Question]
    B[Generate Answer]
    C[Answer Grader]
    D{Correct Answer?}
    E[Final Answer]
    F[Regenerate Answer]

    A --> B
    B --> C
    C --> D
    D -->|YES| E
    D -->|NO| F
```

## Example

**Question**

```text
What documents are required for the First Graduate Scholarship?
```

↓

**Generated Answer**

```text
Income certificate, Community certificate and Marksheet.
```

↓

**Answer Grader**

```text
YES ✅
```

---

**Generated Answer**

```text
The capital of India is New Delhi.
```

↓

**Answer Grader**

```text
NO ❌
```

In [18]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

# State
class AgentState(TypedDict):
    question: str
    answer: str
    grade: str

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Generate Answer
def generate(state: AgentState):

    response = llm.invoke(state["question"])

    return {
        "answer": response.content
    }

# Answer Grader
def answer_grader(state: AgentState):

    prompt = f"""
You are an answer grader.

Question:
{state["question"]}

Answer:
{state["answer"]}

Does the answer correctly answer the user's question?

Answer only YES or NO.
"""

    response = llm.invoke(prompt)

    return {
        "grade": response.content.strip()
    }

# Build Graph
builder = StateGraph(AgentState)

builder.add_node("generate", generate)
builder.add_node("answer_grader", answer_grader)

builder.add_edge(START, "generate")
builder.add_edge("generate", "answer_grader")
builder.add_edge("answer_grader", END)

graph = builder.compile()

# Run
result = graph.invoke({
    "question": "What is the eligibility criteria for the First Graduate Scholarship?"
})

print("Answer:\n", result["answer"])
print("\nGrade:", result["grade"])

Answer:
 The eligibility criteria for the First Graduate Scholarship may vary depending on the specific scholarship program and the organization offering it. However, here are some common eligibility criteria for First Graduate Scholarships:

1. **First-generation college student**: The applicant must be a first-generation college student, meaning that neither of their parents has a bachelor's degree.
2. **Academic achievement**: The applicant must have a strong academic record, with a minimum GPA requirement (e.g., 3.0 or higher).
3. **Enrollment in a degree program**: The applicant must be enrolled in a degree program at an accredited college or university.
4. **Financial need**: The applicant must demonstrate financial need, as determined by the Free Application for Federal Student Aid (FAFSA) or other financial aid forms.
5. **Residency requirements**: Some scholarships may require the applicant to be a resident of a specific state or region.
6. **Citizenship requirements**: The ap

# Multi-Tool Agent

> **Multi-Tool Agent** என்பது, User கேள்வியைப் பொறுத்து சரியான Tool-ஐ தேர்வு செய்து பயன்படுத்தும் AI Agent.

```mermaid
flowchart TD

    A[User Question]
    B[Multi-Tool Agent]
    C[Calculator]
    D[Web Search]
    E[LLM]
    F[Final Answer]

    A --> B
    B -->|Math| C
    B -->|Latest News| D
    B -->|General Question| E
    C --> F
    D --> F
    E --> F
```

## Examples

### Calculator

```text
25 + 75
```

➡️ Uses **Calculator Tool**

---

### Web Search

```text
Latest AI News
```

➡️ Uses **Tavily Web Search**

---

### General Question

```text
What is Machine Learning?
```

➡️ Uses **Groq LLM**

In [27]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

# State
class AgentState(TypedDict):
    question: str
    answer: str

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Web Search Tool
web_search = TavilySearch(max_results=3)

# Calculator Tool
def calculator(expression: str):
    return eval(expression)

# Multi Tool Agent
def multi_tool_agent(state: AgentState):

    question = state["question"].lower()

    # Calculator Tool
    if any(op in question for op in ["+", "-", "*", "/"]):
        result = calculator(question)
        return {"answer": f"Calculator Result: {result}"}

    # Web Search Tool
    elif "latest" in question or "news" in question:
        results = web_search.invoke(question)
        return {"answer": str(results)}

    # LLM Tool
    else:
        response = llm.invoke(question)
        return {"answer": response.content}


# Build Graph
builder = StateGraph(AgentState)

builder.add_node("multi_tool_agent", multi_tool_agent)

builder.add_edge(START, "multi_tool_agent")
builder.add_edge("multi_tool_agent", END)

graph = builder.compile()


# Test
result = graph.invoke({
    "question": "What is the latest AI news?"
})

# Test Calculator
result = graph.invoke({
    "question": "2*75"
})

print(result)
print(result["answer"])

print(result["answer"])

{'question': '2*75', 'answer': 'Calculator Result: 150'}
Calculator Result: 150
Calculator Result: 150


# MemorySaver

> **MemorySaver** என்பது, AI-யுடன் நடந்த **Conversation-ஐ (Question & Answer)** தற்காலிகமாக **Save** செய்து வைக்கும் LangGraph Component.

---

## Workflow

```mermaid
flowchart TD

    A[User Question]
    B[LLM]
    C[Answer]
    D[MemorySaver]

    A --> B
    B --> C
    C --> D
```

---

## Example

### First Question

```text
User: My name is Suresh.
AI: Hello Suresh!
```

↓

**MemorySaver**

```text
Conversation Saved ✅
```

---

### Second Question

```text
User: What is my name?
```

↓

**Expected Answer**

```text
Your name is Suresh.
```

↓

**Actual Output**

```text
I don't know your name.
```

### Why?

```mermaid
flowchart TD

    A[User: My name is Suresh]
    B[MemorySaver]
    C[Conversation Saved]
    D[LLM]
    E[Answer: Hello Suresh]

    A --> B
    B --> C
    A --> D
    D --> E

    F[User: What is my name?]
    G[LLM]
    H[Answer: I don't know your name]

    F --> G
    G --> H

    C -. Saved Conversation NOT Sent to LLM .-> G
```

---

## Explanation

> **MemorySaver** conversation-ஐ **save மட்டும்** பண்ணும்.

> அந்த **saved conversation-ஐ LLM-க்கு அனுப்புற code-ஐ** நாம எழுதவில்லை.

> அதனால் **LLM பழைய conversation-ஐ பார்க்க முடியாது.**

> **Production Chatbot-ல்** `MessagesState` மற்றும் **message history**-யை LLM-க்கு pass பண்ணும்போதுதான் **Memory உண்மையாக வேலை செய்யும்.**

In [35]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq

# State
class AgentState(TypedDict):
    question: str
    answer: str

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Chat Node
def chatbot(state: AgentState):

    response = llm.invoke(state["question"])

    return {
        "answer": response.content
    }

# Build Graph
builder = StateGraph(AgentState)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# Memory
memory = MemorySaver()

# Compile with Memory
graph = builder.compile(checkpointer=memory)

# Thread ID
config = {
    "configurable": {
        "thread_id": "user_1"
    }
}

# First Question
result = graph.invoke(
    {
        "question": "My name is Suresh."
    },
    config=config
)

print(result["answer"])

# Second Question
result = graph.invoke(
    {
        "question": "What is my name?"
    },
    config=config
)

print(result["answer"])

Hello Suresh! It's nice to meet you. Is there something I can help you with or would you like to chat?
I don't know your name. I'm a large language model, I don't have the ability to know your personal information or recall previous conversations. Each time you interact with me, it's a new conversation. If you'd like to share your name, I'd be happy to chat with you!


# Human-in-the-Loop

> **Human-in-the-Loop (HITL)** என்பது, AI ஒரு Action செய்யும் முன் அல்லது Final Answer கொடுக்கும் முன் **Human Approval** பெறும் முறை.

---

## Workflow

```mermaid
flowchart TD

    A[User Question]
    B[Human Approval]
    C{Approved?}
    D[Generate Answer]
    E[Reject Request]

    A --> B
    B --> C
    C -->|Yes| D
    C -->|No| E
```

---

## Example

### User Question

```text
What is the scholarship amount?
```

↓

### Human Approval

```text
Approve this question? (yes/no)
```

↓

If the user enters:

```text
yes
```

↓

```text
Answer Generated ✅
```

---

If the user enters:

```text
no
```

↓

```text
Request Rejected ❌
```

---

## Explanation

> **Human-in-the-Loop** என்பது, AI தானாக முடிவு எடுக்காமல், ஒரு மனிதரின் அனுமதியை (Approval) பெற்ற பிறகே அடுத்த Action-ஐ செய்யும் முறையாகும்.

### Real-World Examples

- Bank Loan Approval
- Medical Diagnosis
- Legal Document Review
- HR Resume Screening
- Email Approval before Sending

In [47]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# State
class AgentState(TypedDict):
    question: str
    approved: bool
    answer: str


# Human Approval Node
def human_approval(state: AgentState):

    approval = input("Approve this question? (yes/no): ")

    return {
        "approved": approval.lower() == "yes"
    }


# Generate Node
def generate(state: AgentState):

    return {
        "answer": f"Answer: {state['question']}"
    }


# Reject Node
def reject(state: AgentState):

    return {
        "answer": "Request Rejected by Human."
    }


# Decision
def decide(state: AgentState):

    if state["approved"]:
        return "generate"

    return "reject"


# Build Graph
builder = StateGraph(AgentState)

builder.add_node("human_approval", human_approval)
builder.add_node("generate", generate)
builder.add_node("reject", reject)

builder.add_edge(START, "human_approval")

builder.add_conditional_edges(
    "human_approval",
    decide,
    {
        "generate": "generate",
        "reject": "reject"
    }
)

builder.add_edge("generate", END)
builder.add_edge("reject", END)

graph = builder.compile()


# Run
result = graph.invoke({
    "question": "What is the scholarship amount?"
})

print(result["answer"])

Request Rejected by Human.


# Multi-Agent Systems

> **Multi-Agent System** என்பது, ஒரு பெரிய Task-ஐ ஒரு AI மட்டும் செய்யாமல், பல AI Agents சேர்ந்து வேலை செய்வது.

---

## Workflow

```mermaid
flowchart TD

    A[User Question]
    B[Research Agent]
    C[Writer Agent]
    D[Final Answer]

    A --> B
    B --> C
    C --> D
```

---

## Example

### User Question

```text
What is the scholarship amount?
```

↓

### Research Agent

```text
Finds the scholarship information.
```

↓

### Writer Agent

```text
Creates a clear final answer.
```

↓

### Final Answer

```text
Scholarship amount is ₹50,000.
```

---

## Explanation

> **Multi-Agent Systems** என்பது, ஒவ்வொரு Agent-க்கும் தனித்தனி பொறுப்பு (Responsibility) இருக்கும். ஒவ்வொரு Agent தனது வேலையை முடித்த பிறகு, அடுத்த Agent-க்கு தகவலை அனுப்பும். இதனால் பெரிய மற்றும் சிக்கலான Tasks-ஐ எளிதாகவும் திறமையாகவும் முடிக்க முடியும்.

### Real-World Examples

- Research Agent → தகவல் சேகரிக்கும்
- Writer Agent → Final Answer எழுதும்
- Reviewer Agent → Answer சரிபார்க்கும்
- Planner Agent → அடுத்த Steps திட்டமிடும்

In [48]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# State
class AgentState(TypedDict):
    question: str
    research: str
    answer: str

# Research Agent
def research_agent(state: AgentState):

    return {
        "research": "Scholarship amount is ₹50,000."
    }

# Writer Agent
def writer_agent(state: AgentState):

    return {
        "answer": f"Final Answer:\n{state['research']}"
    }

# Build Graph
builder = StateGraph(AgentState)

builder.add_node("research_agent", research_agent)
builder.add_node("writer_agent", writer_agent)

builder.add_edge(START, "research_agent")
builder.add_edge("research_agent", "writer_agent")
builder.add_edge("writer_agent", END)

graph = builder.compile()

# Run
result = graph.invoke({
    "question": "What is the scholarship amount?"
})

print(result["answer"])

Final Answer:
Scholarship amount is ₹50,000.


# Production Agentic RAG

> **Production Agentic RAG** என்பது, பல AI Agents ஒன்றாக இணைந்து, User கேள்விக்கு மிகவும் துல்லியமான (Accurate) மற்றும் நம்பகமான (Reliable) பதிலை வழங்கும் முழுமையான AI Workflow ஆகும்.

---

## Workflow

```mermaid
flowchart TD

    A[User Question]
    B[Router Agent]
    C[Query Rewriting]
    D[Retrieve Documents]
    E[Document Grader]
    F[Web Search]
    G[Generate Answer]
    H[Hallucination Checker]
    I[Answer Grader]
    J[Final Answer]

    A --> B
    B --> C
    C --> D
    D --> E
    E --> F
    F --> G
    G --> H
    H --> I
    I --> J
```

---

## Production Pipeline

1. **Router Agent** → எந்த Source பயன்படுத்த வேண்டும் என்று முடிவு செய்கிறது.
2. **Query Rewriting** → User Question-ஐ Search-க்கு ஏற்றவாறு மாற்றுகிறது.
3. **Retrieve** → Vector Database-லிருந்து Documents எடுக்கிறது.
4. **Document Grader** → தொடர்புடைய Documents மட்டும் தேர்வு செய்கிறது.
5. **Web Search** → தேவையானால் இணையத்தில் இருந்து புதிய தகவலை பெறுகிறது.
6. **Generate Answer** → LLM Final Answer உருவாக்குகிறது.
7. **Hallucination Checker** → Answer Documents-க்கு பொருந்துகிறதா என்று சரிபார்க்கிறது.
8. **Answer Grader** → User Question-க்கு சரியான பதிலா என்று மதிப்பீடு செய்கிறது.
9. **Final Answer** → பயனருக்கு நம்பகமான பதில் வழங்கப்படுகிறது.

---

## Real-World Example

**User Question**

```text
What documents are required for the First Graduate Scholarship?
```

↓

**Router Agent**

```text
Use Vector Database
```

↓

**Query Rewriting**

```text
What are the required documents for the First Graduate Scholarship?
```

↓

**Retrieve**

```text
Fetch Top 5 Documents
```

↓

**Document Grader**

```text
Keep Only Relevant Documents
```

↓

**Generate Answer**

```text
Income Certificate, Community Certificate, Marksheet...
```

↓

**Hallucination Checker**

```text
YES
```

↓

**Answer Grader**

```text
YES
```

↓

**Final Answer**

```text
Required documents:
• Income Certificate
• Community Certificate
• Marksheet
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# State
class AgentState(TypedDict):
    question: str
    documents: list
    answer: str


# Router
def router(state):
    print("Router Agent")
    return state


# Query Rewriting
def rewrite_query(state):
    print("Query Rewriting")
    return state


# Retrieve
def retrieve(state):
    print("Retrieve Documents")
    return {
        "documents": ["Document 1", "Document 2"]
    }


# Document Grader
def document_grader(state):
    print("Document Grader")
    return state


# Web Search
def web_search(state):
    print("Web Search")
    return state


# Generate
def generate(state):
    print("Generate Answer")
    return {
        "answer": "Scholarship Answer"
    }


# Hallucination Checker
def hallucination_checker(state):
    print("Hallucination Checker")
    return state


# Answer Grader
def answer_grader(state):
    print("Answer Grader")
    return state


# Build Graph
builder = StateGraph(AgentState)

builder.add_node("router", router)
builder.add_node("rewrite_query", rewrite_query)
builder.add_node("retrieve", retrieve)
builder.add_node("document_grader", document_grader)
builder.add_node("web_search", web_search)
builder.add_node("generate", generate)
builder.add_node("hallucination_checker", hallucination_checker)
builder.add_node("answer_grader", answer_grader)

builder.add_edge(START, "router")
builder.add_edge("router", "rewrite_query")
builder.add_edge("rewrite_query", "retrieve")
builder.add_edge("retrieve", "document_grader")
builder.add_edge("document_grader", "web_search")
builder.add_edge("web_search", "generate")
builder.add_edge("generate", "hallucination_checker")
builder.add_edge("hallucination_checker", "answer_grader")
builder.add_edge("answer_grader", END)

graph = builder.compile()

result = graph.invoke({
    "question": "What is the First Graduate Scholarship?"
})

print(result["answer"])

Router Agent
Query Rewriting
Retrieve Documents
Document Grader
Web Search
Generate Answer
Hallucination Checker
Answer Grader
Scholarship Answer
